# 24. Multi-Channel Spectrum Prediction: GCN + LSTM

Implements the **GCN+LSTM** multi-channel spectrum prediction from:
**"Multi-Channel Spectrum Prediction Algorithm Based on GCN and LSTM"** (IEEE VTC 2022-Fall).

## Paper (adapted to 72h→24h)
- **Graph:** Each channel (band) = node; edges = Spearman correlation between channels (top-K per node). Adjacency A from training data.
- **GCN (2 layers):** Learns frequency-domain (channel) correlation via graph convolution (symmetric normalized Laplacian).
- **LSTM (2 layers):** Learns time-domain correlation. Cascade: graph-structured input → GCN → LSTM → output.
- **Data:** Multi-channel aligned: input (72, N), output (24, N) with N = number of bands.

## Same setup as 10–23
Data: work_dir/final, 72h→24h. Metrics: MAE, RMSE, MASE. Naive baseline. Same visuals.


**GPU:** Use kernel **Python 3.11 (cablelabs-3 .venv)** (Kernel → Change kernel) so TensorFlow uses the project's venv with tensorflow-metal.

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
print(f'TensorFlow: {tf.__version__}')


TensorFlow: 2.18.1


In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus, 'GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU enabled: {len(gpus)} device(s)')
        with tf.device('/GPU:0'): _ = tf.constant(1)
        print('GPU ready.')
    except RuntimeError as e: print('GPU config:', e)
else: print('No GPU found. Change kernel to "Python 3.11 (cablelabs-3 .venv)" (Kernel → Change kernel), then re-run from the top.')
USE_GPU = len(gpus) > 0


GPU enabled: 1 device(s)
GPU ready.


2026-02-10 00:05:19.591519: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-02-10 00:05:19.591549: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-02-10 00:05:19.591557: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
I0000 00:00:1770699919.591570 42557129 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1770699919.591589 42557129 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## Data loading: multi-channel aligned (72, N) → (24, N)

In [3]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not final_dir.exists() or not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError("work_dir/final/training and testing not found")
class_options = sorted([d.name for d in training_dir.iterdir() if d.is_dir()])
LOOKBACK = 72
FORECAST_HORIZON = 24
print(f"Bands: {class_options}, Lookback={LOOKBACK}, Horizon={FORECAST_HORIZON}")

def load_data_for_band(band_name: str, split: str):
    split_dir = final_dir / split / band_name
    if not split_dir.exists(): return pd.DataFrame()
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        try: dfs.append(pd.read_parquet(p))
        except Exception as e: print(f"Error loading {p}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def build_multichannel_sequences(train_dfs, test_dfs, bands, lookback=72, horizon=24):
    # Align all bands by (date, hour); one row per time, columns = bands
    merged_tr = None
    for band in bands:
        df = train_dfs[band]
        if df.empty: continue
        df = df.sort_values(["date", "hour"]).drop_duplicates(["date", "hour"])
        ths = sorted(df["threshold_dbm"].unique())
        if len(ths) > 1: df = df[df["threshold_dbm"] == ths[0]]
        key = df[["date", "hour"]].drop_duplicates()
        au = df.groupby(["date", "hour"])["au_pct"].mean().reset_index()
        au = au.rename(columns={"au_pct": band})
        merged_tr = au if merged_tr is None else pd.merge(merged_tr, au, on=["date", "hour"], how="inner")
    if merged_tr is None or len(merged_tr) < lookback + horizon:
        return None, None, None, None
    band_cols = [b for b in bands if b in merged_tr.columns]
    N = len(band_cols)
    mat_tr = merged_tr[band_cols].values.astype(np.float32)
    merged_te = None
    for band in bands:
        df = test_dfs[band]
        if df.empty: continue
        df = df.sort_values(["date", "hour"])
        ths = sorted(df["threshold_dbm"].unique())
        if len(ths) > 1: df = df[df["threshold_dbm"] == ths[0]]
        au = df.groupby(["date", "hour"])["au_pct"].mean().reset_index()
        au = au.rename(columns={"au_pct": band})
        merged_te = au if merged_te is None else pd.merge(merged_te, au, on=["date", "hour"], how="inner")
    for c in band_cols:
        if c not in merged_te.columns:
            merged_te[c] = np.nan
    merged_te = merged_te[band_cols]
    merged_te = merged_te.ffill().fillna(0)
    mat_te = merged_te.values.astype(np.float32) if len(merged_te) > 0 else np.zeros((0, N))
    X_tr, y_tr = [], []
    for i in range(len(mat_tr) - lookback - horizon + 1):
        X_tr.append(mat_tr[i:i+lookback])
        y_tr.append(mat_tr[i+lookback:i+lookback+horizon])
    X_te, y_te = [], []
    n_test_days = len(mat_te) // horizon
    for d in range(n_test_days):
        if d == 0:
            inp = mat_tr[-lookback:] if len(mat_tr) >= lookback else np.vstack([np.zeros((lookback - len(mat_tr), N)), mat_tr])
        else:
            h = max(0, lookback - d * horizon)
            if h > 0:
                inp = np.vstack([mat_tr[-h:], mat_te[:d*horizon]])
            else:
                inp = mat_te[d*horizon - lookback:d*horizon]
        tgt = mat_te[d*horizon:(d+1)*horizon]
        if len(inp) == lookback and len(tgt) == horizon:
            X_te.append(inp)
            y_te.append(tgt)
    if not X_tr or not X_te:
        return None, None, None, None
    return np.array(X_tr), np.array(y_tr), np.array(X_te), np.array(y_te), band_cols


Bands: ['195MHz', '2441MHz', '3765MHz', '539MHz', '5500MHz', '915MHz'], Lookback=72, Horizon=24


In [4]:
train_data_by_band = {}
test_data_by_band = {}
for band in class_options:
    tr = load_data_for_band(band, "training")
    te = load_data_for_band(band, "testing")
    if not tr.empty and not te.empty:
        train_data_by_band[band] = tr
        test_data_by_band[band] = te
bands_used = [b for b in class_options if b in train_data_by_band and b in test_data_by_band]
out = build_multichannel_sequences(train_data_by_band, test_data_by_band, bands_used, LOOKBACK, FORECAST_HORIZON)
if out[0] is None:
    raise ValueError("Multi-channel sequences could not be built (align bands by date/hour).")
X_train, y_train, X_test, y_test, band_cols = out
N_CHANNELS = len(band_cols)
print(f"Train: {X_train.shape}, Test: {X_test.shape}, Channels: {N_CHANNELS} {band_cols}")

Train: (49, 72, 6), Test: (3, 72, 6), Channels: 6 ['195MHz', '2441MHz', '3765MHz', '539MHz', '5500MHz', '915MHz']


## Adjacency matrix from Spearman correlation (top-K per node)

In [5]:
TOP_K = min(9, max(1, N_CHANNELS - 1))
mat_flat = X_train.reshape(-1, N_CHANNELS)
rho = np.zeros((N_CHANNELS, N_CHANNELS))
for i in range(N_CHANNELS):
    for j in range(N_CHANNELS):
        r, _ = spearmanr(mat_flat[:, i], mat_flat[:, j])
        rho[i, j] = r if not np.isnan(r) else 0
np.fill_diagonal(rho, 0)
A = np.zeros_like(rho)
for i in range(N_CHANNELS):
    idx = np.argsort(rho[i])[-TOP_K:]
    A[i, idx] = np.maximum(rho[i, idx], 0)
A = (A + A.T) / 2
A = np.float32(np.maximum(A, 0))
print(f'Adjacency built (Spearman top-K={TOP_K}). Nonzeros: {np.count_nonzero(A)}')

Adjacency built (Spearman top-K=5). Nonzeros: 16


## GCN layer (symmetric normalized) + 2-layer GCN + 2-layer LSTM

In [6]:
def normalize_adj(A):
    A = A + np.eye(A.shape[0], dtype=np.float32)
    d = np.sum(A, axis=1)
    d_inv_sqrt = np.power(np.maximum(d, 1e-8), -0.5)
    return np.float32(np.diag(d_inv_sqrt) @ A @ np.diag(d_inv_sqrt))

A_norm = normalize_adj(A)
A_tf = tf.constant(A_norm, dtype=tf.float32)

class GCNLayer(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1], self.units), initializer='glorot_uniform', trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        # inputs (batch, N, F), A (N, N)
        out = tf.matmul(inputs, self.w)
        out = tf.matmul(A_tf, out)
        return tf.nn.relu(out)

def build_gcn_lstm(N, T_in, T_out, gcn_units=64, lstm_units=128):
    inp = layers.Input(shape=(T_in, N, 1))
    # (batch, T_in, N, 1) -> apply GCN at each t -> (batch, T_in, N, gcn_units)
    gcn1 = GCNLayer(gcn_units)
    gcn2 = GCNLayer(gcn_units)
    x_list = []
    for t in range(T_in):
        xt = inp[:, t, :, :]
        xt = gcn1(xt)
        xt = gcn2(xt)
        x_list.append(xt)
    x = tf.stack(x_list, axis=1)
    x = tf.reshape(x, [-1, T_in, N * gcn_units])
    x = layers.LSTM(lstm_units, return_sequences=True)(x)
    x = layers.LSTM(lstm_units, return_sequences=False)(x)
    x = layers.Dense(T_out * N, activation='linear')(x)
    out = tf.reshape(x, [-1, T_out, N])
    return keras.Model(inp, out)

model = build_gcn_lstm(N_CHANNELS, LOOKBACK, FORECAST_HORIZON)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
model.summary()

ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


## Train and evaluate

In [ ]:
scaler_x = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))
X_train_flat = X_train.reshape(-1, N_CHANNELS)
X_test_flat = X_test.reshape(-1, N_CHANNELS)
y_train_flat = y_train.reshape(-1, N_CHANNELS)
scaler_x.fit(X_train_flat)
scaler_y.fit(y_train_flat)
X_train_s = scaler_x.transform(X_train_flat).reshape(X_train.shape)
X_test_s = scaler_x.transform(X_test_flat).reshape(X_test.shape)
y_train_s = scaler_y.transform(y_train_flat).reshape(y_train.shape)
X_train_s = X_train_s[:, :, :, np.newaxis]
X_test_s = X_test_s[:, :, :, np.newaxis]

BATCH = 64 if USE_GPU else 32
EPOCHS = 50
history = model.fit(X_train_s, y_train_s, epochs=EPOCHS, batch_size=BATCH, validation_split=0.2, verbose=1)

y_pred_s = model.predict(X_test_s, verbose=0)
y_pred_gcn = scaler_y.inverse_transform(y_pred_s.reshape(-1, N_CHANNELS)).reshape(y_test.shape)
y_pred_gcn = np.clip(y_pred_gcn, 0, 100).astype(np.float32)

def naive_predictor_multi(X, horizon):
    last = X[:, -1, :]
    return np.tile(last[:, np.newaxis, :], (1, horizon, 1))
y_pred_naive = naive_predictor_multi(X_test, FORECAST_HORIZON)

def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
def calculate_mase(y_true, y_pred, y_train):
    mae = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    scale = np.mean(np.abs(np.diff(y_train.flatten()))) if len(y_train.flatten()) > 1 else 1.0
    return mae / max(scale, 1e-8)

mae_g = calculate_mae(y_test, y_pred_gcn)
rmse_g = calculate_rmse(y_test, y_pred_gcn)
mase_g = calculate_mase(y_test, y_pred_gcn, y_train)
mae_n = calculate_mae(y_test, y_pred_naive)
rmse_n = calculate_rmse(y_test, y_pred_naive)
mase_n = calculate_mase(y_test, y_pred_naive, y_train)

results_df = pd.DataFrame([
    {"Model": "GCN+LSTM", "MAE": mae_g, "RMSE": rmse_g, "MASE": mase_g},
    {"Model": "Naive Baseline", "MAE": mae_n, "RMSE": rmse_n, "MASE": mase_n},
])
results_df["MAE"] = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["MASE"] = results_df["MASE"].round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h, Channels: {N_CHANNELS}")
print(f"Test samples: {len(y_test)}")
print(results_df.to_string(index=False))
display(results_df)

## Analysis and Visualizations (same as notebook 10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MASE']
colors = ['#2ecc71', '#3498db', '#9b59b6']
for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].values
    bars = ax.bar(results_df['Model'], vals, color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=15)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02 * max(vals), f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.suptitle('Multi-Channel Prediction: GCN + LSTM', y=1.02, fontsize=12)
plt.show()

naive_mae = results_df[results_df['Model'] == 'Naive Baseline']['MAE'].values[0]
naive_rmse = results_df[results_df['Model'] == 'Naive Baseline']['RMSE'].values[0]
naive_mase = results_df[results_df['Model'] == 'Naive Baseline']['MASE'].values[0]
improvement = results_df[results_df['Model'] != 'Naive Baseline'].copy()
improvement['MAE_imp_%'] = (1 - improvement['MAE'] / naive_mae) * 100
improvement['RMSE_imp_%'] = (1 - improvement['RMSE'] / naive_rmse) * 100
improvement['MASE_imp_%'] = (1 - improvement['MASE'] / naive_mase) * 100
print('Improvement over Naive Baseline (%):')
display(improvement[['Model', 'MAE_imp_%', 'RMSE_imp_%', 'MASE_imp_%']].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(improvement))
w = 0.25
ax.bar(x - w, improvement['MAE_imp_%'], w, label='MAE', color='#2ecc71')
ax.bar(x, improvement['RMSE_imp_%'], w, label='RMSE', color='#3498db')
ax.bar(x + w, improvement['MASE_imp_%'], w, label='MASE', color='#9b59b6')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(improvement['Model'], rotation=15)
ax.set_ylabel('Improvement (%)')
ax.legend()
ax.set_title('Improvement over Naive (positive = better)')
plt.tight_layout()
plt.show()

In [ ]:
hours = np.arange(FORECAST_HORIZON)
ch_show = 0
n_show = min(3, len(y_test))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1: axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_test[i, :, ch_show], 'k--', linewidth=2.5, label='Actual', alpha=0.8)
    ax.plot(hours, y_pred_gcn[i, :, ch_show], '-', linewidth=1.6, label='GCN+LSTM')
    ax.plot(hours, y_pred_naive[i, :, ch_show], '-', linewidth=1.2, label='Naive', alpha=0.7)
    ax.set_title(f'Test sample {i+1}, channel {band_cols[ch_show]}')
    ax.set_xlabel('Hour')
    ax.set_ylabel('AU (%)')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_test.mean(axis=0).mean(axis=1), 'k--', linewidth=2.5, label='Actual (mean)')
ax.plot(hours, y_pred_gcn.mean(axis=0).mean(axis=1), '-', linewidth=1.6, label='GCN+LSTM (mean)')
ax.plot(hours, y_pred_naive.mean(axis=0).mean(axis=1), '-', linewidth=1.2, label='Naive (mean)', alpha=0.7)
ax.set_title('Mean 24h profile (avg over channels)')
ax.set_xlabel('Hour')
ax.set_ylabel('AU (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

mae_per_hour = np.abs(y_test - y_pred_gcn).mean(axis=(0, 2))
mae_per_hour_n = np.abs(y_test - y_pred_naive).mean(axis=(0, 2))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, mae_per_hour, '-o', label='GCN+LSTM', markersize=4)
ax.plot(hours, mae_per_hour_n, '-o', label='Naive', markersize=4, alpha=0.7)
ax.set_title('MAE by forecast hour (avg over channels)')
ax.set_xlabel('Hour')
ax.set_ylabel('MAE (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
residuals_g = (y_test - y_pred_gcn).flatten()
residuals_naive = (y_test - y_pred_naive).flatten()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals_g, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Count')
axes[0].set_title('Residuals: GCN+LSTM')
axes[1].hist(residuals_naive, bins=50, color='#95a5a6', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals: Naive')
plt.suptitle('Residual distribution', y=1.02)
plt.tight_layout()
plt.show()
print(f'Residual mean (bias): GCN+LSTM = {residuals_g.mean():.4f}, Naive = {residuals_naive.mean():.4f}')
print(f'Residual std:         GCN+LSTM = {residuals_g.std():.4f}, Naive = {residuals_naive.std():.4f}')

best_row = results_df[results_df['Model'] == 'GCN+LSTM'].iloc[0]
imp_mae = (1 - best_row['MAE'] / naive_mae) * 100
print(f"\nBest model: GCN+LSTM (MAE={best_row['MAE']:.4f}, RMSE={best_row['RMSE']:.4f}, MASE={best_row['MASE']:.4f}). Improvement over Naive: MAE {imp_mae:+.1f}%.")

### Key insights

- **Paper (VTC 2022-Fall):** Spectrum data as graph (channels = nodes, edges = Spearman correlation, top-K); 2-layer GCN for channel correlation, 2-layer LSTM for temporal correlation; GCN+LSTM outperforms CNN+LSTM.
- **Multi-channel:** Input (72, N), output (24, N); data built by aligning bands on (date, hour). With one band, N=1 and GCN still applies (single-node graph).
- **Improvement over Naive:** Positive % means GCN+LSTM beats the last-value baseline.
